**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Alignment and Event Reference**

![Alignment and Event Reference](../../images/02_alignment_and_event_reference.svg)

**Objective:** Review original source-channel conventions and time coordinates before constructing independent force-derived initial-contact (IC) and terminal-contact (TC) references. Once the physical reference contract is defensible, prepare participant-disjoint evaluation inputs.

**Current evidence:** The completed Notebook 01 source audit contains **nine source-paired participants (AB06–AB14)** and **27 treadmill IMU/force pairs** with overlapping native `Header` intervals. This confirms source availability and timestamp readability, **not** force-contact reference validity.

**Notebook scope:** Source-contract audit, diagnostic figure, conditional force-event extraction, eligibility/exclusion tables, participant-disjoint roles and downloadable outputs. Until the independent physical contract is verified, downstream stages are clearly deferred rather than assigned artificial results.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
# This IPython-native checkout requires no imports outside Section 01.
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = "/content/human-movement-intelligence"
shell = get_ipython()
if shell is None:
    raise RuntimeError("Execute this notebook in Google Colab or IPython.")
shell.system(
    f'if [ ! -d "{repo_path}" ]; then git clone -q "{repo_url}" '
    f'"{repo_path}"; fi'
)

# ---------------------------- 2. Open Repository -----------------------------
shell.run_line_magic("cd", repo_path)


/content/human-movement-intelligence


**01. Imports and Research Environment**

Use the established shared utilities and Camargo-specific alignment/reference functions. If a dirty existing Colab checkout defers a Git fast-forward, Section 01 can retrieve only the missing source-contract module into an isolated runtime directory without altering tracked files or original data. This recovery is not a complete Git synchronisation. No source dataset download or model training is triggered by these imports.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
from pathlib import Path
import subprocess
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 2

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib
import importlib.util
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import os
import re
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import read_sensor_table, validate_timebase
from gait_alignment import mapped_series, validated_overlap
from gait_reference import force_contact_events
from gait_experiments import save_prepared_trial, read_prepared_trial
from gait_evaluation import participant_roles, development_folds
# ------------ 5A. Non-Destructive Missing-Module Recovery --------------------
# A dirty Colab checkout can defer its Git fast-forward. Never discard
# local notebook outputs or original Camargo recordings to resolve imports.
# Only if this module is absent, fetch its public source into a separate
# temporary runtime directory; the local Git checkout is not modified.
if importlib.util.find_spec("gait_contract_evidence") is None:
    remote_relative = "gait_analysis/code/src/gait_contract_evidence.py"
    runtime_modules = root.parent / ".hmi_gait_runtime_modules"
    runtime_modules.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "fetch", "--no-tags", "origin", "main"],
        cwd=root, check=True, capture_output=True, text=True,
    )
    fetched_module = subprocess.run(
        ["git", "show", f"FETCH_HEAD:{remote_relative}"],
        cwd=root, check=True, capture_output=True,
    )
    runtime_module = runtime_modules / "gait_contract_evidence.py"
    runtime_module.write_bytes(fetched_module.stdout)
    if str(runtime_modules) not in sys.path:
        sys.path.insert(0, str(runtime_modules))
    importlib.invalidate_caches()
    print(
        "Missing Gait module recovered from origin/main into a separate "
        "runtime directory. Local source files and outputs were preserved; "
        "the Git checkout was NOT fast-forwarded."
    )

from gait_contract_evidence import (
    audit_candidate_pairs, plot_source_contract_probe,
    validate_explicit_contract,
)

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get("CAMARGO_DATA_ROOT",
                                  root / "data/raw/camargo"))
output_dir = study_dir / "output"
cache_dir = root / "data/processed/camargo/paired_trials"
output_dir.mkdir(parents=True, exist_ok=True)


**02. Audit Original Sensor Columns, Time Coordinates and Force Direction**

Read the current Notebook 01 `01_candidate_pairs.csv` and `01_observed_columns.csv` and use the per-trial `01_trial_timebase_audit.csv` where available. The executed nine-person audit found 27 readable pairs with positive native-time overlap, IMU median `Header` increments of approximately 0.005 and treadmill force increments of approximately 0.001 (source-time units). The force-channel candidates `Treadmill_R_vy` and `Treadmill_L_vy` were present.

The original [Camargo dataset README](https://data.mendeley.com/datasets/fcgm3chfff/1) states that `Header` timestamps are used to synchronise channels and identifies native rates of 200 Hz for IMU and 1,000 Hz for force plates. This supports inspecting time-base compatibility, **not** treating every overlap as an independently validated millisecond-level alignment. The source also supplies right- and left-side gait-cycle annotations (`gcRight`/`gcLeft`), but those derived labels are **not currently used as an alternative primary reference**.

The executed section generated [`02_source_contract_review.csv`](../../output/02_source_contract_review.csv) and a [recorded signal figure](../../output/02_source_contract_diagnostic.png) showing foot/shank IMU channels and right/left treadmill components. The force channels **exist** and show alternating loading-like profiles; no source file is missing. These are descriptive measurements in original source units. Physical vertical-axis mapping, Newton scaling, force polarity, correspondence to the instrumented foot, and common-clock acceptance still require the source contract to be established. The workflow SVG is schematic; the new PNG is the recorded diagnostic.


In [ ]:
# ==================== 02. Review Original Sensor Contract ====================

# ------------- 1. Load Recorded Source Pair and Channel Evidence -------------
pair_file = output_dir / "01_candidate_pairs.csv"
column_file = output_dir / "01_observed_columns.csv"
timebase_file = output_dir / "01_trial_timebase_audit.csv"
if not pair_file.is_file() or not column_file.is_file():
    raise FileNotFoundError(
        "Notebook 01's candidate pairs and observed columns are missing. "
        "Run Notebook 01 Sections 03–07 before inspecting references."
    )
pairs = pd.read_csv(pair_file)
pairs = pairs.loc[
    pairs.candidate_pair & pairs.condition.eq("treadmill")
].copy()
observed = pd.read_csv(column_file)
if pairs.empty:
    raise ValueError("No source-matched treadmill IMU/force trial pairs.")
imu_names = set(observed.loc[observed.modality == "imu", "column"])
fp_names = set(observed.loc[observed.modality == "fp", "column"])
if timebase_file.is_file():
    timebase_audit = pd.read_csv(timebase_file)
    if len(timebase_audit) != len(pairs):
        print("Trial-level timestamp audit is incomplete; rerun "
              "Notebook 01 Sections 05–07 to refresh the evidence.")
else:
    timebase_audit = pd.DataFrame()
    print("Trial-level timestamp audit missing. Rerun updated "
          "Notebook 01 Section 05 before reference extraction.")
print("Source-paired people:", pairs.subject_id.nunique())
print("Candidate source pairs:", len(pairs))
print("Gyroscope fields:", sorted(
    name for name in imu_names if "Gyro" in name
))
print("Treadmill force fields:", sorted(
    name for name in fp_names
    if "Treadmill_" in name and "_v" in name
))
if not timebase_audit.empty:
    display(timebase_audit)


# ------------------- 2. Save Scientific Contract Evidence --------------------
contract_review = pd.DataFrame([
    {"setting": "GYRO_COLUMN", "candidate": "foot_Gyro_Y",
     "observed": "foot_Gyro_Y" in imu_names,
     "unresolved": "Verify the selected gyroscope axis and instrumented side."},
    {"setting": "VERTICAL_FORCE_COLUMN",
     "candidate": "Treadmill_R_vy",
     "observed": "Treadmill_R_vy" in fp_names,
     "unresolved": "Verify physical vertical axis, source units and belt side."},
    {"setting": "FOOT_SIDE", "candidate": "right (candidate)",
     "observed": pd.NA,
     "unresolved": "Confirm physical IMU foot side and force-plate R/L mapping."},
    {"setting": "FORCE_UPWARD_SIGN", "candidate": "positive (candidate)",
     "observed": pd.NA,
     "unresolved": "Verify the real loading polarity and swing-phase baseline."},
    {"setting": "TIME_UNIT_IN_SECONDS", "candidate": "1.0 (candidate)",
     "observed": pd.NA,
     "unresolved": "Confirm Header time units with original 200/1000-Hz rates."},
    {"setting": "IMU_FORCE_TIMEBASE_CONFIRMED",
     "candidate": "not confirmed", "observed": pd.NA,
     "unresolved": "Confirm a common event-level clock for retained trials."},
    {"setting": "FORCE_SCALE_TO_NEWTONS", "candidate": "not confirmed",
     "observed": pd.NA,
     "unresolved": "Confirm conversion from original force units to newtons."},
])
display(contract_review)
contract_review.to_csv(
    output_dir / "02_source_contract_review.csv", index=False
)


# ------------- 3. Inspect Recorded Right-Left Treadmill Signals --------------
sample = pairs.iloc[0]
imu_path = source_root / sample.imu_path
fp_path = source_root / sample.fp_path
if imu_path.is_file() and fp_path.is_file():
    inspect_imu = read_sensor_table(imu_path)
    inspect_fp = read_sensor_table(fp_path)
    time_imu = validate_timebase(inspect_imu, "IMU")
    time_fp = validate_timebase(inspect_fp, "force")
    fig, axes = plt.subplots(
        2, 1, figsize=(11, 6.5), layout="constrained"
    )
    trial_window_imu = time_imu <= time_imu[0] + 5.0
    trial_window_fp = time_fp <= time_fp[0] + 5.0
    for name, colour in (
        ("foot_Gyro_Y", "#236CDE"),
        ("shank_Gyro_Y", "#168C5B"),
    ):
        if name in inspect_imu:
            axes[0].plot(
                time_imu[trial_window_imu],
                inspect_imu.loc[trial_window_imu, name],
                label=name, color=colour, linewidth=1,
            )
    for name, colour in (
        ("Treadmill_R_vy", "#E87929"),
        ("Treadmill_L_vy", "#236CDE"),
    ):
        if name in inspect_fp:
            axes[1].plot(
                time_fp[trial_window_fp],
                inspect_fp.loc[trial_window_fp, name],
                label=name, color=colour, linewidth=1,
            )
    for ax, title in zip(axes, (
        "Recorded IMU: candidate foot/shank gyroscope",
        "Recorded right/left treadmill force components",
    )):
        ax.set_title(title, loc="left")
        ax.set_xlabel("Original Header (source time units)")
        ax.set_ylabel("Recorded value (source units)")
        if ax.lines:
            ax.legend(frameon=False)
        ax.grid(alpha=0.2)
    fig.suptitle(
        "Source-contract inspection: candidate channels only; "
        "no validated IC/TC labels"
    )
    save_figure(fig, output_dir / "02_source_contract_diagnostic.png")
    plt.show()
    print("Original recorded diagnostic:", sample.subject_id,
          sample.trial_id)
    print("Native timestamp start difference:",
          float(time_imu[0] - time_fp[0]))
    print("Native timestamp end difference:",
          float(time_imu[-1] - time_fp[-1]))
else:
    print("Original sensor files are not present in this Colab runtime. "
          "The saved table audit is available, but a real signal "
          "diagnostic cannot be generated without the source MAT files.")

# ------------------ 4. Retain Unverified Physical Contract -------------------
# These are intentionally unset; source column names are not proof of units.
GYRO_COLUMN = None
VERTICAL_FORCE_COLUMN = None
FOOT_SIDE = None
FORCE_UPWARD_SIGN = None
TIME_UNIT_IN_SECONDS = None
FORCE_SCALE_TO_NEWTONS = None
IMU_FORCE_TIMEBASE_CONFIRMED = False
REFERENCE_THRESHOLD_N = 20.0
ANALYSED_CONDITIONS = {"treadmill"}
print("Descriptive source inspection complete. Contact-event extraction "
      "is still blocked pending physical channel and timebase verification.")
print("Evidence saved to:", output_dir / "02_source_contract_review.csv")


**02A. Cross-Trial Physical Source Evidence**

This source-only stage reads every selected treadmill IMU/force pair and compares original right/left treadmill force components (`vx`, `vy`, `vz`), native timestamps and foot gyroscope axes. It records absent original files rather than replacing them with saved CSVs. All numerical values remain in original source units.

No component is automatically declared vertical, no belt is assigned to a foot, and no IC/TC markers are generated in this inspection. The nominal 200/1,000-Hz sensor rates and `Header` timestamp convention are source descriptions, not independent proof of event-level synchronization.

After reviewing the original files, independently establish the gyro axis, physical force component, foot side, loading sign, force conversion to newtons, timestamp units and common clock. The proposed 20-N contact threshold is meaningful only for genuine Newton-valued force. Sections 03–05 remain deferred until that evidence exists.


In [ ]:
# ===================== 02A. Cross-Trial Source Evidence ======================

# ----------------------- 1. Inspect Every Source Pair ------------------------
trial_evidence, component_evidence = audit_candidate_pairs(
    pairs, source_root, read_sensor_table, validate_timebase
)
trial_evidence.to_csv(
    output_dir / "02_contract_evidence_by_trial.csv", index=False
)
component_evidence.to_csv(
    output_dir / "02_force_component_evidence.csv", index=False
)
print("Original-source pairs listed:", len(trial_evidence))
print("Pairs actually inspected:",
      int(trial_evidence.inspection_status.eq("inspected").sum()))
print("Absent/invalid raw sources:",
      int(trial_evidence.inspection_status.ne("inspected").sum()))
print("No automatic physical-axis, side or timebase acceptance is performed.")
display(trial_evidence)
if not component_evidence.empty:
    display(component_evidence.head(12))

# --------------------- 2. Representative Original Traces ---------------------
inspected_keys = trial_evidence.loc[
    trial_evidence.inspection_status.eq("inspected"),
    ["subject_id", "session_id", "condition", "trial_id"]
]
if not inspected_keys.empty:
    inspected_pairs = pairs.merge(
        inspected_keys, on=["subject_id", "session_id", "condition", "trial_id"],
        validate="one_to_one"
    )
    plot_source_contract_probe(
        inspected_pairs.iloc[0], source_root,
        read_sensor_table, validate_timebase,
        output_dir / "02_all_force_axes_source_probe.png"
    )
    print("Source-only multi-axis figure saved.")
else:
    print("Source-only figure deferred: original MAT files are unavailable.")


**02B. Apply Only an Independently Documented Source Contract**

The fields below remain unset until source metadata, sensor documentation and the actual traces support their physical interpretation. Set `force_scale_to_newtons` explicitly, even if the verified factor is `1.0`, and record the source in `original_source_note`. A nonempty note is not itself scientific verification.

Never infer the clock offset from IMU-to-reference agreement on reserved Test data. After establishing the reference contract, rerun 02A, 02B and Sections 03–06 in the same environment containing original MAT recordings. If any field is unresolved, deferred reference extraction is the correct behaviour.


In [ ]:
# ================== 02B. Explicit Physical Source Contract ===================

# ----------------------- 1. Document Physical Mapping ------------------------
# Replace None only after independent source and signal review.
reviewed_contract = {
    "gyro_column": None,                 # Reviewed foot/shank gyroscope axis
    "force_column": None,                # Physically vertical force channel
    "foot_side": None,                   # right or left, physically matched
    "force_upward_sign": None,           # +1 or -1, physically established
    "time_unit_in_seconds": None,        # Original Header conversion
    "force_scale_to_newtons": None,      # Original force to physical newtons
    "imu_force_timebase_confirmed": False,
    "original_source_note": None,        # Exact source evidence, not a guess
}

# ---------------------- 2. Validate the Supplied Values ----------------------
# In a re-run, never reuse values retained from an earlier source contract.
GYRO_COLUMN = None
VERTICAL_FORCE_COLUMN = None
FOOT_SIDE = None
FORCE_UPWARD_SIGN = None
TIME_UNIT_IN_SECONDS = None
FORCE_SCALE_TO_NEWTONS = None
IMU_FORCE_TIMEBASE_CONFIRMED = False
try:
    source_contract = validate_explicit_contract(
        gyro_column=reviewed_contract["gyro_column"],
        force_column=reviewed_contract["force_column"],
        foot_side=reviewed_contract["foot_side"],
        force_upward_sign=reviewed_contract["force_upward_sign"],
        time_unit_in_seconds=reviewed_contract["time_unit_in_seconds"],
        force_scale_to_newtons=reviewed_contract["force_scale_to_newtons"],
        imu_force_timebase_confirmed=reviewed_contract[
            "imu_force_timebase_confirmed"
        ],
        verified_original_source_note=reviewed_contract[
            "original_source_note"
        ],
        observed_imu_columns=imu_names,
        observed_force_columns=fp_names,
    )
except ValueError as source_error:
    print("Physical source contract remains unresolved:", source_error)
    print("Sections 03–05 will remain deferred; this is expected.")
else:
    GYRO_COLUMN = source_contract["GYRO_COLUMN"]
    VERTICAL_FORCE_COLUMN = source_contract["VERTICAL_FORCE_COLUMN"]
    FOOT_SIDE = source_contract["FOOT_SIDE"]
    FORCE_UPWARD_SIGN = source_contract["FORCE_UPWARD_SIGN"]
    TIME_UNIT_IN_SECONDS = source_contract["TIME_UNIT_IN_SECONDS"]
    FORCE_SCALE_TO_NEWTONS = source_contract["FORCE_SCALE_TO_NEWTONS"]
    IMU_FORCE_TIMEBASE_CONFIRMED = source_contract[
        "IMU_FORCE_TIMEBASE_CONFIRMED"
    ]
    print("Explicit source mapping supplied. Inspect scientific provenance "
          "before interpreting downstream contact events.")



**03. Construct Independent Force-Contact References When Validated**

A source-matched trial is not automatically an eligible force-contact reference. This stage uses the documented physical and temporal contract only once the gyroscope channel, vertical-force component, foot side, loading sign, time conversion and common clock have been independently justified.

**Executed status:** The physical source-contract settings remain unresolved, so this stage correctly reported **DEFERRED** and did not produce `02_reference_events.csv` or numerical caches. The Force signal exists; the unresolved issue is scientific mapping and independently checked event timing, not absent Force recordings. This prevents a predictable guard exception during a source-audit-only run without weakening the scientific requirement. Once a reviewed contract is supplied, the original force-contact algorithm and exclusion logic run unchanged.


In [ ]:
# ====================== 03. Construct Force References =======================

# -------------------- 1. Check Source-Contract Readiness ---------------------
missing = [
    name for name, value in {
        "GYRO_COLUMN": GYRO_COLUMN,
        "VERTICAL_FORCE_COLUMN": VERTICAL_FORCE_COLUMN,
        "FOOT_SIDE": FOOT_SIDE if FOOT_SIDE in {"right", "left"} else None,
        "FORCE_UPWARD_SIGN": (
            FORCE_UPWARD_SIGN if FORCE_UPWARD_SIGN in (-1, 1) else None
        ),
        "TIME_UNIT_IN_SECONDS": TIME_UNIT_IN_SECONDS,
        "IMU_FORCE_TIMEBASE_CONFIRMED": IMU_FORCE_TIMEBASE_CONFIRMED,
        "FORCE_SCALE_TO_NEWTONS": FORCE_SCALE_TO_NEWTONS,
    }.items() if value is None or value is False
]
reference_ready = not missing
if not reference_ready:
    print("Independent force-reference extraction DEFERRED:",
          ", ".join(missing))
    print("Review 02_source_contract_review.csv and the original signal "
          "diagnostic; do not interpret this as an executed reference.")
    eligible = pd.DataFrame()
    excluded = pd.DataFrame()
    references = pd.DataFrame()
else:
    if not np.isfinite(float(TIME_UNIT_IN_SECONDS)) or TIME_UNIT_IN_SECONDS <= 0:
        raise ValueError("Time unit conversion must be positive and finite.")
    if GYRO_COLUMN not in imu_names or VERTICAL_FORCE_COLUMN not in fp_names:
        raise ValueError("Reviewed sensor channel absent from original data audit.")
    missing_files = [
        source_root / relative
        for row in pairs.itertuples(index=False)
        for relative in (row.imu_path, row.fp_path)
        if not (source_root / relative).is_file()
    ]
    if missing_files:
        raise FileNotFoundError(
            f"{len(missing_files)} original paired files are missing from "
            f"{source_root}. Use the same Colab runtime as Notebook 01, or "
            "rerun its acquisition step to restore source data."
        )

    # ---------------------- 2. Extract Force References ----------------------
    recorded, exclusions, event_rows = [], [], []
    used_keys = set()
    for row in pairs.itertuples(index=False):
        key = "_".join(re.sub(r"[^A-Za-z0-9]+", "_", str(v)).strip("_")
                       for v in (row.subject_id, row.session_id,
                                 row.condition, row.trial_id))
        if key in used_keys:
            raise ValueError(f"Duplicate trial identity: {key}")
        used_keys.add(key)
        try:
            imu = read_sensor_table(source_root / row.imu_path)
            fp = read_sensor_table(source_root / row.fp_path)
            ti = validate_timebase(imu, "IMU") * float(TIME_UNIT_IN_SECONDS)
            tf = validate_timebase(fp, "force") * float(TIME_UNIT_IN_SECONDS)
            a, b = validated_overlap(ti, tf)
            gyro = mapped_series(imu, GYRO_COLUMN, "gyroscope")
            force = (mapped_series(fp, VERTICAL_FORCE_COLUMN, "vertical force")
                     * FORCE_UPWARD_SIGN * FORCE_SCALE_TO_NEWTONS)
            valid_imu = (ti >= a) & (ti <= b)
            valid_force = (tf >= a) & (tf <= b)
            events = force_contact_events(
                tf[valid_force], force[valid_force],
                threshold_n=REFERENCE_THRESHOLD_N
            )
            ic = events.loc[events.event_type == "IC", "t_reference_s"].to_numpy(float)
            if len(ic) < 3:
                raise ValueError("Too few complete independent force contacts.")
            save_prepared_trial(cache_dir, key, ti[valid_imu],
                                gyro[valid_imu], ic)
            recorded.append(dict(
                subject_id=row.subject_id, session_id=row.session_id,
                condition=row.condition, trial_id=row.trial_id, trial_key=key,
                side=FOOT_SIDE, gyro_channel=GYRO_COLUMN,
                vertical_force_channel=VERTICAL_FORCE_COLUMN,
                n_reference_ic=len(ic), duration_s=b-a
            ))
            for ev in events.itertuples(index=False):
                event_rows.append(dict(trial_key=key,
                    subject_id=row.subject_id, event_type=ev.event_type,
                    t_reference_s=ev.t_reference_s, side=FOOT_SIDE))
        except (ValueError, FileNotFoundError, KeyError) as error:
            exclusions.append(dict(subject_id=row.subject_id,
                session_id=row.session_id, condition=row.condition,
                trial_id=row.trial_id, exclusion_reason=str(error)))
    eligible = pd.DataFrame(recorded, columns=[
        "subject_id","session_id","condition","trial_id","trial_key",
        "side","gyro_channel","vertical_force_channel","n_reference_ic","duration_s"])
    excluded = pd.DataFrame(exclusions, columns=[
        "subject_id","session_id","condition","trial_id","exclusion_reason"])
    references = pd.DataFrame(event_rows, columns=[
        "trial_key","subject_id","event_type","t_reference_s","side"])

    # ----------------------- 3. Save Scientific Tables -----------------------
    eligible.to_csv(output_dir / "02_eligible_trials.csv", index=False)
    excluded.to_csv(output_dir / "02_exclusions.csv", index=False)
    references.to_csv(output_dir / "02_reference_events.csv", index=False)
    display(eligible.head(), excluded.head())
    print("Accepted trials:", len(eligible),
          "Independent force IC events:", int((references.event_type == "IC").sum()))
    if eligible.empty:
        raise ValueError(
            "No aligned eligible IMU/force trials. Review 02_exclusions.csv; "
            "do not calculate detector timing error without independent references."
        )


**04. Assign Participant-Disjoint Development and Test Roles**

This stage operates **only on verified, force-reference-eligible participants**. After valid events exist, reserve whole people with the approved fixed seed and create person-disjoint Development folds. The observed nine source-paired participants are **not** automatically nine force-reference-eligible people. With an unresolved source contract, no partition or model-selection result is generated.


In [ ]:
# ======================= 04. Assign Participant Roles ========================

if not reference_ready:
    print("Stage not executed: independently verified force-contact "
          "references are not available.")
else:

    # ------------------- 1. Stage Inputs and Main Analysis -------------------
    people = eligible.subject_id.unique()
    if len(people) < 7:
        print("Source pilot complete for", len(people),
              "people. Download additional valid participants before grouped "
              "Development/Test assignment and Notebooks 04–05.")
        roles = pd.DataFrame(columns=["subject_id","role"])
        folds = pd.DataFrame(columns=["subject_id","validation_fold"])
    else:
        roles = participant_roles(people, test_fraction=0.2, seed=42)
        folds = development_folds(roles, n_splits=5)
        print("Role counts:", roles.role.value_counts().to_dict())
        display(roles, folds)

    # ----------------------- 2. Save Scientific Tables -----------------------
    roles.to_csv(output_dir / "02_participant_roles.csv", index=False)
    folds.to_csv(output_dir / "02_development_folds.csv", index=False)


**05. Inspect a Source-Aligned Trial**

After successful independent reference extraction, visualise a measured IMU gyroscope trace with force-derived IC timestamps for an eligible trial. These contact markers must not be drawn from unverified force channels or fitted using the IMU predictions themselves. When the reference contract is unresolved, this stage is deferred and no alignment figure is saved.


In [ ]:
# ========================= 05. Inspect Aligned Trial =========================

if not reference_ready:
    print("Stage not executed: independently verified force-contact "
          "references are not available.")
else:

    # ------------------- 1. Stage Inputs and Main Analysis -------------------
    sample = eligible.iloc[0]
    t, g, ic = read_prepared_trial(cache_dir, sample.trial_key)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t, g, label=f"IMU: {GYRO_COLUMN}", lw=1)
    for j, event in enumerate(ic):
        ax.axvline(event, color="darkorange", alpha=0.3,
                   label="Force-derived IC" if j == 0 else None)
    ax.set(xlabel="Reviewed original time (s)",
           ylabel="Gyroscope (source unit)", title="Eligible paired trial")
    ax.legend()

    # ---------------------- 2. Export Scientific Figure ----------------------
    fig.tight_layout()
    save_figure(fig, output_dir / "02_imu_reference_alignment.png")
    plt.show()
    print("Prepared numerical caches:", cache_dir)
    print("Output reference events:", output_dir / "02_reference_events.csv")


**06. Research Summary and Output Download**

The published [`02_source_contract_review.csv`](../../output/02_source_contract_review.csv) and [`02_source_contract_diagnostic.png`](../../output/02_source_contract_diagnostic.png) document original sensor inspection and **do not imply validated IC/TC events**. The `02_eligible_trials.csv`, `02_reference_events.csv`, `02_participant_roles.csv`, and alignment figure become scientific results **only after** the validated reference pathway is actually executed.

The final cell packages current `02_` research files for download from Colab. It does not package raw Camargo archives or automatically commit files to GitHub.


In [ ]:
# ================= 06. Research Summary and Output Download ==================

print('Reference extraction executed:', bool(reference_ready))
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
